## SpendWise AI - Notebook 07: Recommendation Engine

**Objective:** Generate personalized savings recommendations

**Features:** Overspending detection by category; subscription optimization; high-frequency purchase alerts; budget recommendations; positive reinforcement for good habits.

**What you'll learn:** Rule-based recommendation systems; combining multiple signals; prioritizing recommendations.

In [14]:
import json
from pathlib import Path
from datetime import datetime, timedelta
from dataclasses import dataclass, field
from typing import List, Optional
from enum import Enum
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT / "src").exists():
    pass
elif PROJECT_ROOT.name == "src" and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
elif (PROJECT_ROOT / "spendwise-ai" / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT / "spendwise-ai"
print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: /Users/patel/Documents/E/MPS Sem 5/EAI 6020/Final_Project/spendwise-ai


### 2. Load Data

In [15]:
transactions_df = pd.read_csv(PROJECT_ROOT / "data/synthetic/transactions_full.csv")
transactions_df["date"] = pd.to_datetime(transactions_df["date"])
transactions_df["month"] = transactions_df["date"].dt.to_period("M")
transactions_df["week"] = transactions_df["date"].dt.to_period("W")
transactions_df["day_of_week"] = transactions_df["date"].dt.day_name()
transactions_df["is_expense"] = transactions_df["amount"] < 0
print(f"Loaded {len(transactions_df):,} transactions")

Loaded 122,754 transactions


### 3. Define Recommendation Data Structures

In [16]:
class Priority(Enum):
    HIGH = "high"
    MEDIUM = "medium"
    LOW = "low"
    POSITIVE = "positive"

class RecommendationType(Enum):
    OVERSPENDING = "overspending"
    SUBSCRIPTION = "subscription"
    FREQUENCY = "frequency"
    TREND = "trend"
    BUDGET = "budget"
    POSITIVE = "positive"

@dataclass
class Recommendation:
    type: RecommendationType
    priority: Priority
    title: str
    description: str
    potential_savings: float = 0.0
    category: Optional[str] = None
    action_items: List[str] = field(default_factory=list)

    def to_dict(self) -> dict:
        return {"type": self.type.value, "priority": self.priority.value, "title": self.title, "description": self.description, "potential_savings": self.potential_savings, "category": self.category, "action_items": self.action_items}

sample_rec = Recommendation(type=RecommendationType.OVERSPENDING, priority=Priority.HIGH, title="High Food Delivery Spending", description="Your food delivery spending is 45% above your average", potential_savings=120.0, category="Food & Dining", action_items=["Try meal prepping on weekends", "Set a weekly food delivery budget"])
print("Sample recommendation:", json.dumps(sample_rec.to_dict(), indent=2))

Sample recommendation: {
  "type": "overspending",
  "priority": "high",
  "title": "High Food Delivery Spending",
  "description": "Your food delivery spending is 45% above your average",
  "potential_savings": 120.0,
  "category": "Food & Dining",
  "action_items": [
    "Try meal prepping on weekends",
    "Set a weekly food delivery budget"
  ]
}


### 4. Build Analysis Functions

In [17]:
class SpendingAnalyzer:
    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()
        self.expenses = df[df["is_expense"]].copy()
        self.expenses["amount"] = self.expenses["amount"].abs()

    def get_category_stats(self, user_id: str) -> pd.DataFrame:
        user_expenses = self.expenses[self.expenses["user_id"] == user_id]
        monthly = user_expenses.groupby(["month", "category"])["amount"].sum().unstack(fill_value=0)
        stats = pd.DataFrame({"mean": monthly.mean(), "std": monthly.std(), "min": monthly.min(), "max": monthly.max(), "last_month": monthly.iloc[-1] if len(monthly) > 0 else 0, "prev_month": monthly.iloc[-2] if len(monthly) > 1 else 0})
        stats["change_pct"] = ((stats["last_month"] - stats["prev_month"]) / stats["prev_month"] * 100).fillna(0)
        stats["vs_average_pct"] = ((stats["last_month"] - stats["mean"]) / stats["mean"] * 100).fillna(0)
        return stats

    def get_subscription_analysis(self, user_id: str) -> dict:
        user_expenses = self.expenses[self.expenses["user_id"] == user_id]
        merchant_stats = user_expenses.groupby("merchant").agg({"amount": ["count", "mean", "std"], "category": "first", "subcategory": "first"})
        merchant_stats.columns = ["count", "avg_amount", "std_amount", "category", "subcategory"]
        subscriptions = merchant_stats[(merchant_stats["count"] >= 3) & (merchant_stats["std_amount"] < 2)].copy()
        by_category = subscriptions.groupby("category").agg({"avg_amount": ["sum", "count"]})
        by_category.columns = ["total", "count"]
        duplicates = by_category[by_category["count"] > 1]
        return {"subscriptions": subscriptions.to_dict("index"), "total_monthly": subscriptions["avg_amount"].sum(), "duplicate_categories": duplicates.to_dict("index"), "count": len(subscriptions)}

    def get_frequency_analysis(self, user_id: str) -> dict:
        user_expenses = self.expenses[self.expenses["user_id"] == user_id]
        max_date = user_expenses["date"].max()
        recent = user_expenses[user_expenses["date"] > max_date - timedelta(days=30)]
        freq = recent.groupby("subcategory").agg({"amount": ["count", "sum", "mean"]})
        freq.columns = ["count", "total", "avg_per_transaction"]
        freq = freq.sort_values("count", ascending=False)
        high_freq = freq[freq["count"] > 10]
        return {"high_frequency": high_freq.to_dict("index"), "all_frequencies": freq.head(10).to_dict("index")}

    def get_income_expense_ratio(self, user_id: str) -> dict:
        user_data = self.df[self.df["user_id"] == user_id]
        max_date = user_data["date"].max()
        last_month = user_data[user_data["date"] > max_date - timedelta(days=30)]
        income = last_month[last_month["amount"] > 0]["amount"].sum()
        expenses = last_month[last_month["amount"] < 0]["amount"].abs().sum()
        savings_rate = (income - expenses) / income * 100 if income > 0 else 0
        return {"income": income, "expenses": expenses, "net": income - expenses, "savings_rate": savings_rate, "expense_ratio": expenses / income * 100 if income > 0 else 100}

In [18]:
analyzer = SpendingAnalyzer(transactions_df)
test_user = "user_0001"
print("Testing Analyzer:\n")
cat_stats = analyzer.get_category_stats(test_user)
print("Category Statistics (top 5):\n", cat_stats[["last_month", "mean", "vs_average_pct"]].head())
subs = analyzer.get_subscription_analysis(test_user)
print(f"\nSubscriptions: {subs['count']} found, ${subs['total_monthly']:.2f}/month")
freq = analyzer.get_frequency_analysis(test_user)
print(f"High-frequency categories: {len(freq['high_frequency'])}")
ratio = analyzer.get_income_expense_ratio(test_user)
print(f"Savings rate: {ratio['savings_rate']:.1f}%")

Testing Analyzer:

Category Statistics (top 5):
                    last_month         mean  vs_average_pct
category                                                  
Bills & Utilities      229.50   644.785714      -64.406780
Education             5285.34  2495.892857      111.761494
Entertainment          924.66   803.798571       15.036283
Financial             2184.88  2940.832857      -25.705400
Food & Dining          788.88  1016.121429      -22.363610

Subscriptions: 3 found, $19.61/month
High-frequency categories: 2
Savings rate: -413.5%


### 5. Build Recommendation Engine

In [19]:
def _get_category_tips(category: str) -> List[str]:
    tips = {"Food & Dining": ["Try meal prepping on weekends", "Use grocery pickup to avoid impulse buys", "Set a weekly dining out budget"], "Transportation": ["Consider carpooling or public transit", "Combine errands to reduce trips"], "Shopping": ["Wait 24 hours before non-essential purchases", "Use price tracking tools"], "Entertainment": ["Look for free local events", "Use library resources"], "Subscriptions": ["Audit subscriptions quarterly", "Share family plans when possible"]}
    return tips.get(category, ["Review spending in this category", "Set a monthly budget"])

In [20]:
class RecommendationEngine:
    OVERSPEND_THRESHOLD = 25
    HIGH_FREQ_THRESHOLD = 12
    LOW_SAVINGS_THRESHOLD = 10
    SUBSCRIPTION_DUPLICATE_THRESHOLD = 2

    def __init__(self, df: pd.DataFrame):
        self.analyzer = SpendingAnalyzer(df)

    def generate_recommendations(self, user_id: str) -> List[Recommendation]:
        recs = []
        recs.extend(self._check_overspending(user_id))
        recs.extend(self._check_subscriptions(user_id))
        recs.extend(self._check_high_frequency(user_id))
        recs.extend(self._check_positive_trends(user_id))
        recs.extend(self._check_budget_health(user_id))
        priority_order = {Priority.HIGH: 0, Priority.MEDIUM: 1, Priority.LOW: 2, Priority.POSITIVE: 3}
        recs.sort(key=lambda x: (priority_order[x.priority], -x.potential_savings))
        return recs

    def _check_overspending(self, user_id: str) -> List[Recommendation]:
        recs = []
        stats = self.analyzer.get_category_stats(user_id)
        for category, row in stats.iterrows():
            if row["vs_average_pct"] > self.OVERSPEND_THRESHOLD:
                excess = row["last_month"] - row["mean"]
                priority = Priority.HIGH if row["vs_average_pct"] > 50 else Priority.MEDIUM if row["vs_average_pct"] > 30 else Priority.LOW
                recs.append(Recommendation(type=RecommendationType.OVERSPENDING, priority=priority, title=f"High {category} Spending", description=f"Your {category} spending is {row['vs_average_pct']:.0f}% above average (${row['last_month']:.2f} vs ${row['mean']:.2f} typical)", potential_savings=round(excess, 2), category=category, action_items=_get_category_tips(category)))
        return recs

    def _check_subscriptions(self, user_id: str) -> List[Recommendation]:
        recs = []
        analysis = self.analyzer.get_subscription_analysis(user_id)
        for category, data in analysis["duplicate_categories"].items():
            if data["count"] >= self.SUBSCRIPTION_DUPLICATE_THRESHOLD:
                subs_in_cat = [(n, i["avg_amount"]) for n, i in analysis["subscriptions"].items() if i["category"] == category]
                potential_savings = min(s[1] for s in subs_in_cat)
                recs.append(Recommendation(type=RecommendationType.SUBSCRIPTION, priority=Priority.MEDIUM, title=f"Multiple {category} Subscriptions", description=f"You have {data['count']} subscriptions in {category} totaling ${data['total']:.2f}/month", potential_savings=round(potential_savings, 2), category=category, action_items=["Review if you need all services", "Consider consolidating", "Check for family/bundle plans"]))
        if analysis["total_monthly"] > 150:
            recs.append(Recommendation(type=RecommendationType.SUBSCRIPTION, priority=Priority.MEDIUM, title="High Total Subscription Cost", description=f"Subscriptions total ${analysis['total_monthly']:.2f}/month (${analysis['total_monthly']*12:.2f}/year)", potential_savings=round(analysis["total_monthly"] * 0.2, 2), action_items=["Audit subscriptions quarterly", "Cancel unused services", "Look for annual billing discounts"]))
        return recs

    def _check_high_frequency(self, user_id: str) -> List[Recommendation]:
        recs = []
        analysis = self.analyzer.get_frequency_analysis(user_id)
        for subcategory, data in analysis["high_frequency"].items():
            if data["avg_per_transaction"] < 20 and data["count"] > self.HIGH_FREQ_THRESHOLD:
                recs.append(Recommendation(type=RecommendationType.FREQUENCY, priority=Priority.LOW, title=f"Frequent {subcategory} Purchases", description=f"You made {data['count']} {subcategory} purchases totaling ${data['total']:.2f}", potential_savings=round(data["total"] * 0.3, 2), category=subcategory, action_items=["Consider reducing frequency", "Set a weekly budget for this category"]))
        return recs

    def _check_positive_trends(self, user_id: str) -> List[Recommendation]:
        recs = []
        stats = self.analyzer.get_category_stats(user_id)
        for category, row in stats.iterrows():
            if row["change_pct"] < -15 and row["prev_month"] > 50:
                savings = row["prev_month"] - row["last_month"]
                recs.append(Recommendation(type=RecommendationType.POSITIVE, priority=Priority.POSITIVE, title=f"Great Job on {category}", description=f"Your {category} spending decreased by {abs(row['change_pct']):.0f}% from last month", potential_savings=0, category=category, action_items=[f"You saved ${savings:.2f} compared to last month", "Keep up the good work!"]))
        return recs

    def _check_budget_health(self, user_id: str) -> List[Recommendation]:
        recs = []
        ratio = self.analyzer.get_income_expense_ratio(user_id)
        if ratio["savings_rate"] < self.LOW_SAVINGS_THRESHOLD:
            recs.append(Recommendation(type=RecommendationType.BUDGET, priority=Priority.HIGH, title="Low Savings Rate", description=f"Your savings rate is {ratio['savings_rate']:.1f}%. Experts recommend at least 20%.", potential_savings=round(ratio["income"] * 0.1, 2), action_items=["Review discretionary spending", "Set up automatic savings transfers", "Create a monthly budget plan"]))
        elif ratio["savings_rate"] > 30:
            recs.append(Recommendation(type=RecommendationType.POSITIVE, priority=Priority.POSITIVE, title="Excellent Savings Rate", description=f"Your savings rate is {ratio['savings_rate']:.1f}% - well above the recommended 20%", potential_savings=0, action_items=["Consider investing your surplus", "You're building great financial habits!"]))
        return recs

### 6. Test Recommendation Engine

In [21]:
engine = RecommendationEngine(transactions_df)
print("Generating Recommendations for user_0001\n" + "=" * 70)
recommendations = engine.generate_recommendations("user_0001")
for i, rec in enumerate(recommendations, 1):
    print(f"\n#{i} [{rec.priority.value.upper()}] {rec.title}")
    print(f"   {rec.description}")
    if rec.potential_savings > 0:
        print(f"   Potential savings: ${rec.potential_savings:.2f}/month")
    for action in (rec.action_items or [])[:2]:
        print(f"      - {action}")
print("=" * 70)
print(f"Total recommendations: {len(recommendations)}")
total_savings = sum(r.potential_savings for r in recommendations)
print(f"Total potential savings: ${total_savings:.2f}/month (${total_savings*12:.2f}/year)")

Generating Recommendations for user_0001

#1 [HIGH] High Education Spending
   Your Education spending is 112% above average ($5285.34 vs $2495.89 typical)
   Potential savings: $2789.45/month
      - Review spending in this category
      - Set a monthly budget

#2 [HIGH] Low Savings Rate
   Your savings rate is -413.5%. Experts recommend at least 20%.
   Potential savings: $350.70/month
      - Review discretionary spending
      - Set up automatic savings transfers

#3 [MEDIUM] High Health & Wellness Spending
   Your Health & Wellness spending is 30% above average ($1446.50 vs $1109.35 typical)
   Potential savings: $337.15/month
      - Review spending in this category
      - Set a monthly budget

#4 [MEDIUM] Multiple Transportation Subscriptions
   You have 3 subscriptions in Transportation totaling $19.61/month
   Potential savings: $4.69/month
      - Review if you need all services
      - Consider consolidating

#5 [LOW] Frequent Coffee Shops Purchases
   You made 15 Coffee S

### 7. Test Multiple Users

In [22]:
print("Recommendations Summary for Multiple Users\n")
print(f"{'User':<12} {'High':>6} {'Medium':>8} {'Low':>6} {'Positive':>10} {'Savings':>12}")
print("-" * 60)
for user_id in [f"user_{i:04d}" for i in range(1, 11)]:
    recs = engine.generate_recommendations(user_id)
    counts = {p: 0 for p in Priority}
    for r in recs:
        counts[r.priority] += 1
    total_savings = sum(r.potential_savings for r in recs)
    print(f"{user_id:<12} {counts[Priority.HIGH]:>6} {counts[Priority.MEDIUM]:>8} {counts[Priority.LOW]:>6} {counts[Priority.POSITIVE]:>10} ${total_savings:>10.2f}")

Recommendations Summary for Multiple Users

User           High   Medium    Low   Positive      Savings
------------------------------------------------------------
user_0001         2        2      2          7 $   3571.64
user_0002         1        2      1          4 $   1164.42
user_0003         0        3      1          8 $    444.19
user_0004         3        4      2          4 $   9049.40
user_0005         0        0      2         10 $     90.30
user_0006         2        2      3          7 $   7951.22
user_0007         1        0      4          7 $   1763.82
user_0008         1        2      2          8 $   1934.22
user_0009         1        3      2          9 $   1988.82
user_0010         0        1      3          7 $    345.34


In [23]:
class RecommendationService:
    def __init__(self, transactions_df: pd.DataFrame):
        self.engine = RecommendationEngine(transactions_df)

    def get_recommendations(self, user_id: str, limit: int = 10) -> dict:
        recs = self.engine.generate_recommendations(user_id)[:limit]
        by_priority = {p.value: [] for p in Priority}
        for r in recs:
            by_priority[r.priority.value].append(r.to_dict())
        total_savings = sum(r.potential_savings for r in recs)
        return {"user_id": user_id, "total_recommendations": len(recs), "potential_monthly_savings": round(total_savings, 2), "potential_yearly_savings": round(total_savings * 12, 2), "by_priority": by_priority, "recommendations": [r.to_dict() for r in recs]}

    def get_top_recommendation(self, user_id: str) -> dict:
        recs = self.engine.generate_recommendations(user_id)
        return recs[0].to_dict() if recs else None

    def get_savings_summary(self, user_id: str) -> dict:
        recs = self.engine.generate_recommendations(user_id)
        actionable = [r for r in recs if r.priority != Priority.POSITIVE]
        by_category = {}
        for r in actionable:
            if r.category:
                by_category[r.category] = by_category.get(r.category, 0) + r.potential_savings
        return {"user_id": user_id, "total_monthly_savings": round(sum(r.potential_savings for r in actionable), 2), "by_category": {k: round(v, 2) for k, v in sorted(by_category.items(), key=lambda x: -x[1])}, "action_count": len(actionable)}

In [24]:
service = RecommendationService(transactions_df)
print("Testing Recommendation Service\n")
result = service.get_recommendations("user_0001", limit=5)
print(f"User: {result['user_id']}, Total recommendations: {result['total_recommendations']}")
print(f"Potential monthly savings: ${result['potential_monthly_savings']:.2f}, yearly: ${result['potential_yearly_savings']:.2f}")
savings = service.get_savings_summary("user_0001")
print("Savings by category:", list(savings["by_category"].items())[:5])

Testing Recommendation Service

User: user_0001, Total recommendations: 5
Potential monthly savings: $3532.41, yearly: $42388.92
Savings by category: [('Education', np.float64(2789.45)), ('Health & Wellness', np.float64(337.15)), ('Coffee Shops', 50.42), ('Public Transit', 39.23), ('Transportation', 4.69)]


### 9. Format Recommendations for Display

In [25]:
def format_recommendations_text(recommendations: List[dict]) -> str:
    lines = []
    for i, rec in enumerate(recommendations, 1):
        lines.append(f"\n**{rec['title']}**")
        lines.append(f"   {rec['description']}")
        if rec.get("potential_savings", 0) > 0:
            lines.append(f"   Save up to ${rec['potential_savings']:.2f}/month")
        if rec.get("action_items"):
            for action in rec["action_items"][:2]:
                lines.append(f"   -> {action}")
    return "\n".join(lines)

result = service.get_recommendations("user_0001", limit=3)
print(format_recommendations_text(result["recommendations"]))


**High Education Spending**
   Your Education spending is 112% above average ($5285.34 vs $2495.89 typical)
   Save up to $2789.45/month
   -> Review spending in this category
   -> Set a monthly budget

**Low Savings Rate**
   Your savings rate is -413.5%. Experts recommend at least 20%.
   Save up to $350.70/month
   -> Review discretionary spending
   -> Set up automatic savings transfers

**High Health & Wellness Spending**
   Your Health & Wellness spending is 30% above average ($1446.50 vs $1109.35 typical)
   Save up to $337.15/month
   -> Review spending in this category
   -> Set a monthly budget


In [26]:
import nbformat

notebook_path = PROJECT_ROOT / "src" / "07_recommendation_engine.ipynb"
nb = nbformat.read(str(notebook_path), as_version=4)

code_snippets = []
for cell in nb.cells:
    if cell.get("cell_type") == "code":
        src = cell.get("source", "")
        if isinstance(src, list):
            src = "".join(src)
        if (
            "class Priority(" in src
            or "class RecommendationType(" in src
            or "class Recommendation" in src and "dataclass" in src
            or "class SpendingAnalyzer" in src
            or "def _get_category_tips" in src
            or "class RecommendationEngine" in src
            or "class RecommendationService" in src
            or "def format_recommendations_text" in src
        ):
            code_snippets.append(src)

module_code = """from dataclasses import dataclass, field
from datetime import timedelta
from enum import Enum
from typing import List, Optional

import pandas as pd
"""

module_code += "\n\n".join(code_snippets) + "\n"

module_path = PROJECT_ROOT / "src" / "recommendation_engine.py"
module_path.parent.mkdir(parents=True, exist_ok=True)
with open(module_path, "w") as f:
    f.write(module_code)
print(f"Module saved to {module_path}")


Module saved to /Users/patel/Documents/E/MPS Sem 5/EAI 6020/Final_Project/spendwise-ai/src/recommendation_engine.py


### Summary

- Recommendation data structures (Priority, RecommendationType, Recommendation); SpendingAnalyzer (category stats, subscriptions, frequency, income/expense ratio).
- RecommendationEngine: overspending, subscriptions, high-frequency, positive trends, budget health; RecommendationService for production.
- Production module: `src/recommendation_engine.py`. Next: Notebook 08 (Final Pipeline & Streamlit).

In [27]:
# Module saved by the cell above
